# D-10 · ADN tipográfico · Colab de Bruno
Ejecutar en GPU. Clona la rama privada, genera sintético reproducible, mide vecino
antes de entrenar, entrena 20 épocas y guarda pesos, métricas y hashes en Drive.
No lee ni abre el examen REAL. Bruno lo evalúa una vez en evaluar_examen_colab.ipynb con sello de Claude.
La corrida local de una época es únicamente humo; no extrapolar sus cifras.
Leer PRERREGISTRO.md. Requiere acceso al repo privado y espacio en Drive.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, subprocess, sys, getpass, base64, json, datetime, shutil
os.nice(max(0, 15 - os.getpriority(os.PRIO_PROCESS, 0)))
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
drive.mount('/content/drive')
BRANCH = 'studio/adn-tipografico'
REPO = 'https://github.com/brune6320-del/prometeo-studio.git'
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = getpass.getpass('Token GitHub con lectura del repo (no se guarda): ')
assert token, 'Se necesita acceso al repositorio privado'
auth = base64.b64encode(('x-access-token:' + token).encode()).decode()
env = os.environ.copy()
env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader',
           GIT_CONFIG_VALUE_0='Authorization: Basic ' + auth)
CHECKOUT = Path('/content/prometeo-adn')
assert not CHECKOUT.exists(), 'Usar un runtime nuevo para conservar procedencia'
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(CHECKOUT)],env=env,check=True)
del token, auth, env
CODE = CHECKOUT / 'tools/adn_tipo'
COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],cwd=CHECKOUT,text=True).strip()
print('Commit:', COMMIT)
print((CODE/'PRERREGISTRO.md').read_text())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
import torch
assert torch.cuda.is_available(), 'Elegir Entorno de ejecución > GPU. No entrenar completo en CPU.'
print('GPU:', torch.cuda.get_device_name(0))
RUN = Path('/content/drive/MyDrive/Prometeo/ADN') / datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN.mkdir(parents=True, exist_ok=False)
(RUN/'commit.txt').write_text(COMMIT+'\n')
(RUN/'paquetes.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
shutil.copy2(CODE/'PRERREGISTRO.md', RUN/'PRERREGISTRO.md')
# Tarifa de Bruno en USD/h; None conserva coste desconocido en vez de inventar 0.
COSTE_HORA_USD = None
# Carpeta separada creada por Claude con preparar_examen.py --modo train.
NATIVO_TRAIN = None  # p. ej. '/content/drive/MyDrive/Prometeo/nativo-train'
def run(script, *args):
    cmd = [sys.executable, str(CODE/script), *map(str,args)]
    subprocess.run(cmd, cwd=CODE, check=True)
DATA = RUN/'sintetico'
run('generar_sintetico.py','--salida',DATA,'--fuentes',RUN/'fuentes-ofl','--n',500)
manifest = json.loads((DATA/'manifest.json').read_text())
HOLDOUT_SHA = manifest['holdout_sha256']
print('SHA256 holdout:', HOLDOUT_SHA)
print('SHA256 galeria (conservar para examen):', manifest['galeria_sha256'])
run('medir.py',DATA,'--holdout-sha256',HOLDOUT_SHA,'--salida',RUN/'vecino.json')


In [ ]:
coste = [] if COSTE_HORA_USD is None else ['--coste-hora-usd',COSTE_HORA_USD]
run('entrenar.py',DATA,'--salida',RUN/'solo-sintetico','--epocas',20,'--dispositivo','cuda',*coste)
PESOS = RUN/'solo-sintetico/pesos.pt'
run('medir.py',DATA,'--holdout-sha256',HOLDOUT_SHA,'--pesos',PESOS,
    '--dispositivo','cuda','--salida',RUN/'cnn-sintetico.json')
print(json.loads((RUN/'solo-sintetico/entrenamiento.json').read_text()))


In [ ]:
resultados = [RUN/'cnn-sintetico.json']
if NATIVO_TRAIN:
    teacher = Path(NATIVO_TRAIN)
    assert json.loads((teacher/'manifest.json').read_text())['dominio'] == 'nativo_train'
    assert not (teacher/'holdout.json').exists(), 'NO leer el examen real en entrenamiento'
    run('entrenar.py',teacher,'--inicial',PESOS,'--salida',RUN/'afinado-A',
        '--epocas',5,'--dispositivo','cuda',*coste)
    run('medir.py',DATA,'--holdout-sha256',HOLDOUT_SHA,'--pesos',RUN/'afinado-A/pesos.pt',
        '--dispositivo','cuda','--salida',RUN/'cnn-afinado-sintetico.json')
    resultados.append(RUN/'cnn-afinado-sintetico.json')
    print(json.loads((RUN/'afinado-A/entrenamiento.json').read_text()))
else:
    print('Afinado A: NO VERIFICADO; faltan documentos nativos train propios y separados por origen/familia.')
run('comparar.py',RUN/'vecino.json',*resultados,'--salida',RUN/'comparacion-sintetico.json')
print((RUN/'comparacion-sintetico.json').read_text())
print('REAL: NO VERIFICADO. Bruno debe usar evaluar_examen_colab.ipynb una sola vez con el sello de Claude.')
print('Drive:', RUN)


## Examen real separado (Bruno, con sello de Claude)
Conservar los SHA de pesos de ambas etapas y galeria.json al terminar. No abrir
ni listar d10-examen desde este entrenamiento. Claude publica el sello sin nombres
ni contenido y conserva el SHA del prerregistro v2 antes de sellar.
Bruno ejecuta evaluar_examen_colab.ipynb una vez con estos pesos y el catálogo.
Seguir EXAMEN-CLAUDE.md. La salida solo contiene métricas agregadas y hashes.
Si el modelo no gana al vecino, conservar el resultado negativo; no ajustar
pesos, catálogo, umbrales o alias según REAL. Escaneados sin nativo emparejado
no tienen etiquetas de estilo: quedan NO EVALUABLE, no se inventan métricas.
